## Low-cost BGE robustness pilot

Entity + BGE
10%, 30%, 60%

Missing + BGE
10%, 30%, 60%

× 5 Test Configurations
× 20 reps

= $2\times3\times5\times20=600$ metric runs.

\+ shared BGE clean null:

$5\times20=100$ runs.

Total:

$\boxed{700\text{ runs}}$

Verify:
- that the BGE pipeline runs correctly
- that the five tests accept 768D without issues
- that no NaNs or errors appear
- that null and defect paths work
- that changing the encoder does not cause an absurd technical failure
- that the robustness experiment is ready for its subsequent formal run

In [1]:
import importlib,inspect,json,time,numpy as np,pandas as pd
import src.paths as paths,src.metrics as metrics
importlib.reload(paths);importlib.reload(metrics)

from src.paths import get_project_paths

P=get_project_paths()
DATA=P["processed"];RESULTS=P["results"]
BGE=DATA/"robustness_bge"
OUT=RESULTS/"week9"/"bge_operational_pilot"
OUT.mkdir(parents=True,exist_ok=True)

N=100;PILOT_REPS=20;N_PERM=100;ALPHA=.05;BASE_SEED=42
SEVERITIES=[.10,.30,.60]
TESTS=["C2ST-Linear","C2ST-RBF","MMD","Energy","Sinkhorn"]

print("BGE OPERATIONAL PILOT | n=100 | reps=20")

BGE OPERATIONAL PILOT | n=100 | reps=20


In [2]:
def load_bge(prefix):
    X={"clean":np.load(BGE/f"{prefix}_bge_clean.npy")}
    for s in SEVERITIES:X[s]=np.load(BGE/f"{prefix}_bge_{int(s*100)}pct.npy")
    ids=pd.read_csv(BGE/f"{prefix}_bge_campaign_ids.csv")["campaign_id"].astype(str).to_numpy()
    return X,ids

entity,entity_ids=load_bge("entity")
missing,missing_ids=load_bge("missing_reference")

assert np.array_equal(entity_ids,missing_ids)
assert np.array_equal(entity["clean"],missing["clean"])
assert len(entity_ids)==300
assert all(entity[k].shape==(300,768) for k in ["clean",*SEVERITIES])
assert all(missing[k].shape==(300,768) for k in ["clean",*SEVERITIES])
assert all(np.isfinite(entity[k]).all() for k in ["clean",*SEVERITIES])
assert all(np.isfinite(missing[k]).all() for k in ["clean",*SEVERITIES])

print("Entity:",{k:v.shape for k,v in entity.items()})
print("Missing:",{k:v.shape for k,v in missing.items()})
print("Shared clean exact:",np.array_equal(entity["clean"],missing["clean"]))

Entity: {'clean': (300, 768), 0.1: (300, 768), 0.3: (300, 768), 0.6: (300, 768)}
Missing: {'clean': (300, 768), 0.1: (300, 768), 0.3: (300, 768), 0.6: (300, 768)}
Shared clean exact: True


The BGE representations were successfully loaded for both semantic defects. All clean and defective conditions contained 300 campaign-level embeddings with 768 dimensions, and the clean representation was exactly identical across Entity Inconsistency and Missing Cross-Channel Reference. This confirms that the two robustness scenarios share the same frozen clean BGE representation as intended.

In [3]:
with open(BGE/"bge_metric_scales.json") as f:
    cfg=json.load(f)

BGE_SIGMA=float(cfg["mmd_sigma"])
BGE_BLUR=float(cfg["sinkhorn_blur"])

print("BGE MMD sigma:",BGE_SIGMA)
print("BGE Sinkhorn blur:",BGE_BLUR)

assert np.isclose(BGE_SIGMA,0.3873605130049743)
assert np.isclose(BGE_BLUR,0.038736051300497475)

BGE MMD sigma: 0.3873605130049743
BGE Sinkhorn blur: 0.038736051300497475


The representation-specific metric parameters were loaded successfully. **MMD sigma** was fixed at `0.3873605130049743` and **Sinkhorn blur** at `0.038736051300497475`. These values were derived exclusively from the frozen clean BGE reference representation and remained unchanged during the pilot.

In [4]:
sink=[n for n in dir(metrics) if "sinkhorn" in n.lower() and "paired" in n.lower()
      and "permutation" in n.lower() and callable(getattr(metrics,n))]

if len(sink)!=1:
    raise RuntimeError(f"Expected one paired Sinkhorn permutation test, found {sink}")

SINK=getattr(metrics,sink[0])
print("Sinkhorn:",SINK.__name__,inspect.signature(SINK))

def call_filtered(fn,P,Q,**kwargs):
    sig=inspect.signature(fn)
    return fn(P,Q,**{k:v for k,v in kwargs.items() if k in sig.parameters})

def extract(r):
    stat=r.get("accuracy",r.get("statistic",r.get("mmd2",r.get("energy",np.nan))))
    reject=r.get("reject_H0",r.get("reject",float(r["p_value"])<ALPHA))
    return {
        "statistic":float(stat),
        "p_value":float(r["p_value"]),
        "reject":bool(reject),
        "null_mean":r.get("null_mean",np.nan),
        "null_std":r.get("null_std",np.nan)
    }

def run_metric(test,P,Q,seed):
    common={"n_permutations":N_PERM,"seed":seed,"alpha":ALPHA}
    if test=="C2ST-Linear":
        r=call_filtered(metrics.c2st_paired,P,Q,test_size=.5,**common)
    elif test=="C2ST-RBF":
        r=call_filtered(metrics.c2st_rbf_paired,P,Q,test_size=.5,C=1.0,gamma="scale",**common)
    elif test=="MMD":
        r=call_filtered(metrics.mmd_paired_permutation_test,P,Q,sigma=BGE_SIGMA,**common)
    elif test=="Energy":
        r=call_filtered(metrics.energy_paired_permutation_test,P,Q,**common)
    elif test=="Sinkhorn":
        r=call_filtered(SINK,P,Q,blur=BGE_BLUR,**common)
    else:
        raise ValueError(test)
    return extract(r)

Sinkhorn: sinkhorn_paired_permutation_test (P, Q, blur, n_permutations=100, seed=42, alpha=0.05, p=2, backend='tensorized')


In [5]:
rng=np.random.default_rng(BASE_SEED)

sampling_plan={
    rep:rng.choice(300,size=N,replace=False)
    for rep in range(PILOT_REPS)
}

null_rng=np.random.default_rng(BASE_SEED)

null_plan={}
for rep in range(PILOT_REPS):
    selected=null_rng.choice(300,size=2*N,replace=False)
    null_plan[rep]=(selected[:N],selected[N:])

assert all(len(x)==100 for x in sampling_plan.values())
assert all(len(a)==len(b)==100 and len(set(a)&set(b))==0 for a,b in null_plan.values())

print("Power plans:",len(sampling_plan))
print("Null plans:",len(null_plan))

Power plans: 20
Null plans: 20


In [6]:
DATASETS={
    "Entity Inconsistency":entity,
    "Missing Cross-Channel Reference":missing
}

smoke=[]

for scenario,D in DATASETS.items():
    idx=sampling_plan[0]
    for s in SEVERITIES:
        P0=D["clean"][idx];Q0=D[s][idx]
        for test in TESTS:
            t=time.perf_counter()
            try:r=run_metric(test,P0,Q0,BASE_SEED);err=""
            except Exception as e:r={"statistic":np.nan,"p_value":np.nan,"reject":False,"null_mean":np.nan,"null_std":np.nan};err=repr(e)
            smoke.append({"type":"power","scenario":scenario,"severity":s,"test":test,**r,
                          "runtime":time.perf_counter()-t,"error":err})

a,b=null_plan[0]
P0=entity["clean"][a];Q0=entity["clean"][b]

for test in TESTS:
    t=time.perf_counter()
    try:r=run_metric(test,P0,Q0,BASE_SEED);err=""
    except Exception as e:r={"statistic":np.nan,"p_value":np.nan,"reject":False,"null_mean":np.nan,"null_std":np.nan};err=repr(e)
    smoke.append({"type":"null","scenario":"Shared BGE Clean","severity":np.nan,"test":test,**r,
                  "runtime":time.perf_counter()-t,"error":err})

smoke=pd.DataFrame(smoke)

print("Cells:",len(smoke))
print("Errors:",(smoke.error!="").sum())
print("Nonfinite p:",smoke.p_value.isna().sum())

display(smoke[smoke.error!=""])

assert len(smoke)==35
assert (smoke.error=="").all()
assert np.isfinite(smoke.p_value).all()

print("BGE SMOKE TEST PASSED")

Cells: 35
Errors: 0
Nonfinite p: 0


,type,scenario,severity,test,statistic,p_value,reject,null_mean,null_std,runtime,error


BGE SMOKE TEST PASSED


The complete BGE smoke test passed successfully. All 30 defect-power configurations and five shared clean-null configurations executed without errors or non-finite p-values. This confirms that all five test configurations are operationally compatible with the 768-dimensional BGE representation.

In [7]:
records=[]
start=time.perf_counter()

for scenario,D in DATASETS.items():
    for s in SEVERITIES:
        for rep in range(PILOT_REPS):
            idx=sampling_plan[rep]
            P0=D["clean"][idx];Q0=D[s][idx]
            seed=BASE_SEED+rep

            for test in TESTS:
                t=time.perf_counter()
                r=run_metric(test,P0,Q0,seed)
                records.append({
                    "scenario":scenario,"severity":s,"N":N,"test":test,"rep":rep,
                    "seed":seed,**r,"runtime":time.perf_counter()-t
                })

        print(f"{scenario} | {s:.0%} complete")

bge_power=pd.DataFrame(records)

print("Rows:",len(bge_power),"Expected:",600)
print("Runtime:",round((time.perf_counter()-start)/60,2),"min")

assert len(bge_power)==600
assert np.isfinite(bge_power.p_value).all()

Entity Inconsistency | 10% complete
Entity Inconsistency | 30% complete
Entity Inconsistency | 60% complete
Missing Cross-Channel Reference | 10% complete
Missing Cross-Channel Reference | 30% complete
Missing Cross-Channel Reference | 60% complete
Rows: 600 Expected: 600
Runtime: 3.59 min


The low-cost BGE defect-power pilot completed all 600 expected metric runs, corresponding to 30 Scenario × Severity × Test Configuration cells with 20 Monte Carlo repetitions per cell. No incomplete cells or numerical failures were observed. These pilot results are used only to verify execution stability and are not interpreted as confirmatory power estimates.

In [8]:
bge_power.to_csv(OUT/"bge_power_pilot_20reps.csv",index=False)

bge_power_check=bge_power.groupby(
    ["scenario","severity","test"]
).size().reset_index(name="reps")

assert len(bge_power_check)==30
assert bge_power_check.reps.eq(20).all()

print("Saved power cells:",len(bge_power_check))

Saved power cells: 30


In [9]:
null_records=[]
clean=entity["clean"]
start=time.perf_counter()

for rep in range(PILOT_REPS):
    a,b=null_plan[rep]
    P0=clean[a];Q0=clean[b]
    seed=BASE_SEED+rep

    for test in TESTS:
        t=time.perf_counter()
        r=run_metric(test,P0,Q0,seed)
        null_records.append({
            "representation":"BGE Campaign 768D",
            "N":N,"test":test,"rep":rep,"seed":seed,
            **r,"runtime":time.perf_counter()-t
        })

bge_null=pd.DataFrame(null_records)

print("Rows:",len(bge_null),"Expected:",100)
print("Runtime:",round((time.perf_counter()-start)/60,2),"min")

assert len(bge_null)==100
assert np.isfinite(bge_null.p_value).all()

Rows: 100 Expected: 100
Runtime: 0.45 min


The shared BGE clean-null pilot completed all 100 expected runs, corresponding to five test configurations with 20 repetitions each. This verifies that the clean-null calibration path is operational for the BGE representation before the formal robustness analysis.

In [10]:
bge_null.to_csv(OUT/"bge_clean_null_pilot_20reps.csv",index=False)

checks={
    "power_rows":len(bge_power)==600,
    "power_cells":bge_power.groupby(["scenario","severity","test"]).ngroups==30,
    "null_rows":len(bge_null)==100,
    "null_cells":bge_null.groupby(["test"]).ngroups==5,
    "power_finite_p":np.isfinite(bge_power.p_value).all(),
    "null_finite_p":np.isfinite(bge_null.p_value).all(),
    "power_reps_complete":bge_power.groupby(["scenario","severity","test"]).size().eq(20).all(),
    "null_reps_complete":bge_null.groupby(["test"]).size().eq(20).all()
}

display(pd.Series(checks,name="passed"))
assert all(checks.values())

print("BGE OPERATIONAL PILOT COMPLETED SUCCESSFULLY")

power_rows             True
power_cells            True
null_rows              True
null_cells             True
power_finite_p         True
null_finite_p          True
power_reps_complete    True
null_reps_complete     True
Name: passed, dtype: bool

BGE OPERATIONAL PILOT COMPLETED SUCCESSFULLY


In [11]:
runtime_summary=pd.concat([
    bge_power.groupby("test").runtime.mean().rename("power_mean_runtime"),
    bge_null.groupby("test").runtime.mean().rename("null_mean_runtime")
],axis=1)

display(runtime_summary.round(4))
runtime_summary.to_csv(OUT/"bge_pilot_runtime_summary.csv")

,power_mean_runtime,null_mean_runtime
test,,
C2ST-Linear,0.0158,0.0201
C2ST-RBF,0.0206,0.0190
Energy,0.0512,0.0473
MMD,0.0494,0.0438
Sinkhorn,1.6551,1.2064


Runtime remained low for C2ST-Linear, C2ST-RBF, MMD, and Energy Distance, while Sinkhorn was more computationally expensive. This pattern is consistent with the earlier experiments and confirms that Sinkhorn will dominate the computational cost of the formal BGE robustness run. Runtime differences are treated as computational diagnostics only and do not affect metric selection or interpretation.